In [24]:
import pandas as pd
import numpy as np

In [25]:
df = pd.read_csv('/kaggle/input/datasets/mirichoi0218/insurance/insurance.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [26]:
df.isna().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [27]:
df['sex'].unique()

array(['female', 'male'], dtype=object)

In [28]:
df['region'].unique()

array(['southwest', 'southeast', 'northwest', 'northeast'], dtype=object)

In [29]:
df['smoker'].unique()

array(['yes', 'no'], dtype=object)

In [30]:
df['smoker'] = df['smoker'].map({'yes' : 1 , 'no' : 0})

In [31]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,1,southwest,16884.92400
1,18,male,33.770,1,0,southeast,1725.55230
2,28,male,33.000,3,0,southeast,4449.46200
3,33,male,22.705,0,0,northwest,21984.47061
4,32,male,28.880,0,0,northwest,3866.85520


In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder

ohe=OneHotEncoder()
scaler = StandardScaler()

In [33]:
X = df.drop('charges',axis=1)
y=df['charges']

In [34]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.25,random_state=42)

In [35]:
X_train

,age,sex,bmi,children,smoker,region
693,24,male,23.655,0,0,northwest
1297,28,female,26.510,2,0,southeast
634,51,male,39.700,1,0,southwest
1022,47,male,36.080,1,1,southeast
178,46,female,28.900,2,0,southwest
...,...,...,...,...,...,...
1095,18,female,31.350,4,0,northeast
1130,39,female,23.870,5,0,southeast
1294,58,male,25.175,0,0,northeast
860,37,female,47.600,2,1,southwest


In [36]:
y_train

693      2352.96845
1297     4340.44090
634      9391.34600
1022    42211.13820
178      8823.27900
           ...     
1095     4561.18850
1130     8582.30230
1294    11931.12525
860     46113.51100
1126    10214.63600
Name: charges, Length: 1003, dtype: float64

In [37]:
categorical_columns = X.select_dtypes(include='object').columns
numeric_columns = X.select_dtypes(exclude='object').columns

In [38]:
print(categorical_columns)
print(numeric_columns)

Index(['sex', 'region'], dtype='object')
Index(['age', 'bmi', 'children', 'smoker'], dtype='object')


In [39]:
preprocessor = ColumnTransformer(
                [("OneHotEncoder",ohe,categorical_columns),
                ("Standard Scaler",scaler,numeric_columns)],remainder='passthrough'
                )

In [40]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [41]:
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(
    y_train.to_numpy().reshape(-1, 1)
)
y_test_scaled = y_scaler.transform(
    y_test.to_numpy().reshape(-1, 1)
)

In [42]:
import torch
import torch.nn as nn
import torch.optim as optim 

In [43]:
X_train_tensor = torch.tensor(X_train,dtype=torch.float32)
X_test_tensor = torch.tensor(X_test,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled,dtype=torch.float32)

In [44]:
class LinearRegression(nn.Module):
    def __init__(self,input_dim):
        super(LinearRegression,self).__init__()
        
        self.network = nn.Sequential(
                nn.Linear(input_dim,64),
                nn.ReLU(),
                nn.Linear(64,128),
                nn.ReLU(),
                nn.Linear(128,1)
        )

    def forward(self,x):
        return self.network(x)
        

In [45]:
X_train_tensor.shape

torch.Size([1003, 10])

In [46]:
input_dim = X_train_tensor.shape[1]
model = LinearRegression(input_dim)

In [47]:
print(model)

LinearRegression(
  (network): Sequential(
    (0): Linear(in_features=10, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)


In [48]:
## Loss and optimizer

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(),lr = 0.01)

In [49]:
## Training Loop
epochs = 10000
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    predictions = model(X_train_tensor)
    loss = criterion(predictions,y_train_tensor)
    loss.backward()

    optimizer.step()

    if (epoch+1) % 100 == 0:
        print(f"Epoch[{epoch+1}]/{epochs}, Loss : {loss.item():.4f}")
    

Epoch[100]/10000, Loss : 0.1032
Epoch[200]/10000, Loss : 0.0664
Epoch[300]/10000, Loss : 0.0539
Epoch[400]/10000, Loss : 0.0398
Epoch[500]/10000, Loss : 0.0341
Epoch[600]/10000, Loss : 0.0336
Epoch[700]/10000, Loss : 0.0232
Epoch[800]/10000, Loss : 0.0224
Epoch[900]/10000, Loss : 0.0182
Epoch[1000]/10000, Loss : 0.0191
Epoch[1100]/10000, Loss : 0.0160
Epoch[1200]/10000, Loss : 0.0142
Epoch[1300]/10000, Loss : 0.0135
Epoch[1400]/10000, Loss : 0.0172
Epoch[1500]/10000, Loss : 0.0115
Epoch[1600]/10000, Loss : 0.0114
Epoch[1700]/10000, Loss : 0.0200
Epoch[1800]/10000, Loss : 0.0135
Epoch[1900]/10000, Loss : 0.0157
Epoch[2000]/10000, Loss : 0.0110
Epoch[2100]/10000, Loss : 0.0087
Epoch[2200]/10000, Loss : 0.0102
Epoch[2300]/10000, Loss : 0.0079
Epoch[2400]/10000, Loss : 0.0145
Epoch[2500]/10000, Loss : 0.0145
Epoch[2600]/10000, Loss : 0.0091
Epoch[2700]/10000, Loss : 0.0076
Epoch[2800]/10000, Loss : 0.0100
Epoch[2900]/10000, Loss : 0.0101
Epoch[3000]/10000, Loss : 0.0064
Epoch[3100]/10000, 

In [50]:
model.eval()

with torch.no_grad():
    predictions = model(X_test_tensor)

print(predictions[:10])
print(y_test_tensor[:10])

tensor([[-0.0659],
        [-0.8831],
        [ 1.3623],
        [ 1.2892],
        [ 0.7402],
        [-0.9493],
        [-0.2656],
        [-0.0072],
        [-0.6110],
        [-0.2562]])
tensor([[-0.3464],
        [-0.6638],
        [ 1.3335],
        [-0.3293],
        [ 1.7004],
        [-0.7249],
        [-0.9257],
        [ 0.0783],
        [-0.7916],
        [-0.2493]])


In [51]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

y_pred = predictions.numpy()
y_true = y_test_tensor.numpy()

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("RMSE:", rmse)
print("R2:", r2)

RMSE: 0.6986254087273901
R2: 0.5306857824325562


In [52]:
model.eval()

with torch.no_grad():
    predictions_scaled = model(X_test_tensor)


predictions = y_scaler.inverse_transform(
    predictions_scaled.numpy()
)

print(predictions[:10])

[[12474.57  ]
 [ 2631.0989]
 [29677.568 ]
 [28797.068 ]
 [22183.492 ]
 [ 1833.3411]
 [10068.337 ]
 [13181.662 ]
 [ 5908.4463]
 [10182.005 ]]
